# gene-tidy — clean messy gene/protein identifier tables

**Zero setup.** Just run the cells top to bottom.

1. Install gene-tidy
2. Pick a file (a bundled messy example, or upload your own)
3. Run the cleaner
4. Preview the clean / ambiguous / failed rows
5. Download a ZIP with all six output files

gene-tidy maps every identifier to the current **HGNC approved symbol** plus
Ensembl / UniProt / Entrez / RefSeq cross-references, recovers Excel
date-corrupted symbols (`SEPT2 → "2-Sep"`), and **never guesses silently or
drops a row**. It runs **fully offline** against a bundled HGNC dump.

## 1. Install

In [ ]:
# On Colab this installs from PyPI. If gene-tidy isn't on PyPI yet, install
# straight from GitHub by uncommenting the second line.
%pip install -q gene-tidy
# %pip install -q "git+https://github.com/gene-tidy/gene-tidy"

## 2. Choose your input

By default we use the **bundled messy example** so you can click *Run* and see
results immediately. To use your own file, run the upload cell below it.

In [ ]:
from gene_tidy.examples import EXAMPLE_PATH
import pandas as pd

input_path = str(EXAMPLE_PATH)  # the bundled messy_example.xlsx
print('Using bundled example:', input_path)
pd.read_excel(input_path).head(25)

In [ ]:
# OPTIONAL: upload your own .txt / .csv / .tsv / .xlsx instead.
# (Skip this cell to keep using the bundled example.)
try:
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    if uploaded:
        input_path = next(iter(uploaded))
        print('Using uploaded file:', input_path)
except ImportError:
    print('Not on Colab — keeping', input_path)

## 3. Run the cleaner

In [ ]:
from gene_tidy import tidy_file

result = tidy_file(input_path, 'outputs/')

print('HGNC dump:', result.hgnc_version.get('hgnc_release'),
      '(retrieved', result.hgnc_version.get('downloaded_date'), ')')
print('Identifier column(s):', ', '.join(result.id_columns))
print('Counts:', result.counts)

## 4. Preview the results

### ✅ Clean rows (confidently resolved)

In [ ]:
cols = ['input_value', 'detected_type', 'approved_symbol', 'hgnc_id',
        'ensembl_gene_id', 'uniprot_id', 'entrez_id', 'match_status', 'warning']
result.clean[cols]

### ⚠️ Ambiguous rows (need manual review — never auto-resolved)

In [ ]:
result.ambiguous[['input_value', 'detected_type', 'approved_symbol',
                  'match_status', 'warning']]

### ❌ Failed rows (no match)

In [ ]:
result.failed[['input_value', 'detected_type', 'match_status', 'warning']]

### 📝 Methods paragraph (paste into your supplementary methods)

In [ ]:
print(result.methods_text)

## 5. Download all outputs as a ZIP

In [ ]:
import shutil

zip_path = shutil.make_archive('gene_tidy_outputs', 'zip', 'outputs')
print('Wrote', zip_path)

try:
    from google.colab import files  # type: ignore
    files.download(zip_path)
except ImportError:
    print('Not on Colab — find the ZIP and the outputs/ folder in the file browser.')